# Experiment 3 · Go further: the full ethane torsion

Reaction 3a compares only two points: staggered and eclipsed ethane. Here you will compute the **whole torsional profile** by rotating one methyl group from 0° to 120° in 10° steps, with **PySCF** running in your browser.

> Run the cells in order with ▶ (Shift+Enter). The scan takes 2–5 minutes, depending on the method and basis.

In [ ]:
# Install PySCF (takes ~30 s on Colab; skipped if already installed)
try:
    import pyscf
except ImportError:
    %pip install -q pyscf
    import pyscf
print("PySCF", pyscf.__version__)

# Optional: Grimme's D4 dispersion correction, needed only for methods ending in "-D4".
# The ">=1.3" pin matters: older builds of this package break PySCF.
WANT_D4 = True
if WANT_D4:
    %pip install -q "pyscf-dispersion>=1.3"


In [ ]:
import numpy as np
from pyscf import gto, scf, dft

HARTREE_TO_KCAL = 627.5095

def energy(atoms, method, basis, charge=0, mult=1, grid=(99, 590), verbose=False):
    """Single-point energy (hartree) with PySCF.
    method: 'HF', or a functional name such as 'PBE', 'B3LYP', 'wB97M-V';
            add '-D4' for Grimme's D4 dispersion (e.g. 'B3LYP-D4')."""
    mol = gto.M(atom=atoms, basis=basis, charge=charge, spin=mult - 1, verbose=0)
    m = method.upper()
    disp = None
    if m.endswith("-D4"):
        m, disp = m[:-3], "d4"
    if m == "HF":
        mf = scf.RHF(mol) if mult == 1 else scf.UHF(mol)
    else:
        mf = dft.RKS(mol) if mult == 1 else dft.UKS(mol)
        mf.xc = m.replace("-", "_") if m in ("WB97M-V", "B97M-V") else m
        mf.grids.atom_grid = grid
    if disp:
        mf.disp = disp
    mf.conv_tol = 1e-9
    e = mf.kernel()
    if not mf.converged:
        print(f"  WARNING: SCF did not converge for {method}/{basis}")
    if verbose:
        print(f"{method}/{basis}: {e:.10f} Eh, {mol.nao} basis functions")
    return e


In [ ]:
STAGGERED = """
C      0.00000000    -0.00000000    -0.75653070
C      0.00000000     0.00000000     0.75653070
H      0.87676740     0.50620190    -1.15187150
H     -0.00000000    -1.01240380    -1.15187150
H     -0.87676740     0.50620190    -1.15187150
H     -0.87676740    -0.50620190     1.15187150
H      0.87676740    -0.50620190     1.15187150
H     -0.00000000     1.01240380     1.15187150
"""

ECLIPSED = """
C      0.00000000     0.00000000    -0.76368150
C      0.00000000     0.00000000     0.76368150
H      1.00874940     0.00000000    -1.16760390
H     -0.50437470    -0.87360260    -1.16760390
H     -0.50437470     0.87360260    -1.16760390
H      1.00874940     0.00000000     1.16760390
H     -0.50437470     0.87360260     1.16760390
H     -0.50437470    -0.87360260     1.16760390
"""

REF = 2.73  # kcal/mol, eclipsed − staggered (BHROT27)

## Rotating a methyl group
We find the two carbons, pick the three hydrogens bonded to the second carbon, and rotate them rigidly around the C–C axis (Rodrigues' rotation formula). All bond lengths and angles stay fixed. This is a **rigid scan**.

In [ ]:
def parse(xyz):
    rows = [l.split() for l in xyz.strip().split("\n")]
    return [r[0] for r in rows], np.array([[float(x) for x in r[1:4]] for r in rows])

def rotate_methyl(xyz, angle_deg):
    el, X = parse(xyz)
    C = [i for i, e in enumerate(el) if e == "C"]
    c1, c2 = X[C[0]], X[C[1]]
    axis = (c2 - c1) / np.linalg.norm(c2 - c1)
    H2 = [i for i, e in enumerate(el) if e == "H" and np.linalg.norm(X[i] - c2) < 1.3]
    t = np.radians(angle_deg)
    for i in H2:
        v = X[i] - c2
        X[i] = c2 + v*np.cos(t) + np.cross(axis, v)*np.sin(t) + axis*np.dot(axis, v)*(1 - np.cos(t))
    return "\n".join(f"{e} {x:.8f} {y:.8f} {z:.8f}" for e, (x, y, z) in zip(el, X))

print(rotate_methyl(STAGGERED, 60))

## Run the scan

In [ ]:
METHOD, BASIS = "B3LYP-D4", "def2-SVP"   # try "def2-TZVP" (slower), or other methods
angles = np.arange(0, 121, 10)
E = np.array([energy(rotate_methyl(STAGGERED, a), METHOD, BASIS) for a in angles])
rel = (E - E.min()) * HARTREE_TO_KCAL
for a, r in zip(angles, rel):
    print(f"{a:4d}°  {r:6.3f} kcal/mol")

## Fit and plot
A methyl rotor has threefold symmetry, so the simplest model is

$$V(\varphi) = \frac{V_3}{2}\,\big(1 - \cos 3\varphi\big),$$

with φ measured from the staggered conformation and $V_3$ the barrier height.

In [ ]:
import matplotlib.pyplot as plt
phi = np.radians(angles)
V3 = 2 * np.sum(rel * (1 - np.cos(3*phi))) / np.sum((1 - np.cos(3*phi))**2)   # least squares
e_ecl = (energy(ECLIPSED, METHOD, BASIS) - energy(STAGGERED, METHOD, BASIS)) * HARTREE_TO_KCAL
print(f"rigid-scan barrier (60°): {rel[angles == 60][0]:.3f} kcal/mol")
print(f"fitted V3:                {V3:.3f} kcal/mol")
print(f"database geometries:      {e_ecl:.3f} kcal/mol   (reference {REF})")
f = np.linspace(0, 120, 200)
plt.plot(angles, rel, "o", label=f"{METHOD}/{BASIS}")
plt.plot(f, V3/2*(1 - np.cos(np.radians(3*f))), "-", label="V3 fit")
plt.axhline(REF, color="k", lw=1, ls=":", label="reference barrier")
plt.xlabel("H–C–C–H rotation from staggered (°)"); plt.ylabel("relative energy (kcal/mol)"); plt.legend(); plt.show()

## Questions to think about
1. Is the profile well described by the simple $V_3$ cosine? Where are the maxima and minima?
2. The rigid-scan barrier and the barrier from the two database geometries differ slightly. Why? (Hint: compare the C–C bond length in the two database structures.)
3. Change the method (e.g. `HF`, `PBE`, `wB97M-V`) and the basis. Which affects the barrier more for this "easy" reaction?
4. The ethane barrier is often explained by steric repulsion *or* by hyperconjugation. Can a total-energy scan alone tell you which is right?